# 04 — Combined Data Exploration

Exploratory analysis of scraped NBC and Fox News article data. This notebook investigates data completeness, NA patterns, and data quality issues to inform downstream cleaning decisions.

**Datasets:**
- `fox_scraped_all.csv` — scraped Fox News articles
- `nbc_scraped_all.csv` — scraped NBC News articles
- `url_only_data.csv` — external URL reference list

## 0. Imports & Data Loading

In [ ]:
# Put all imports here
import pandas as pd

In [ ]:
# Load data here
fox = pd.read_csv("../data/raw/fox_scraped_all.csv")
nbc = pd.read_csv("../data/raw/nbc_scraped_all.csv")
all_urls = pd.read_csv("../data/external/url_only_data.csv")

In [ ]:
# Check that nbc scraper returned all links that it should have
len(all_urls[all_urls["url"].str.contains("nbcnews.com")]) == len(nbc)

True

## 1. NBC Dataset — NA Exploration

Investigating missing values across `title`, `author`, and `datetime_posted` columns to understand data quality issues and plan cleaning strategies.

In [ ]:
# List NA values by column
nbc.isna().sum()

url                 0
topic               0
title               4
subtitle            4
author             96
datetime_posted    94
label               0
dtype: int64

In [ ]:
# Start with title column (smallest NA count & same as subtitle)
nbc[nbc["title"].isna()]

,url,topic,title,subtitle,author,datetime_posted,label
323,https://www.nbcnews.com/select/shopping/select...,select,NaN,NaN,NaN,NaN,NBC News
354,https://www.nbcnews.com/feature/nbc-out/lil-na...,feature,NaN,NaN,NaN,NaN,NBC News
613,https://www.nbcnews.com/select/shopping/new-no...,select,NaN,NaN,NaN,NaN,NBC News
1786,https://www.nbcnews.com/select/shopping/mr-cof...,select,NaN,NaN,NaN,NaN,NBC News


In [8]:
# Examine actual links manually
broken_urls = nbc[nbc["title"].isna()]["url"].values # All 4 NAs are pages that don't load. -> title and subtitle are both 4 NA so they are the same 4 articles
print(f"Articles with missing titles: {len(broken_urls)}")
print("Conclusion: all are pages that fail to load")
broken_urls

Articles with missing titles: 4
Conclusion: all are pages that fail to load


<StringArray>
[                          'https://www.nbcnews.com/select/shopping/select-pet-awards-under-25-ncna1306012',
 'https://www.nbcnews.com/feature/nbc-out/lil-nas-x-dolly-parton-lena-waithe-appear-virtual-glaad-n1233424',
                          'https://www.nbcnews.com/select/shopping/new-notable-sonos-jbl-fitbit-rcna156060',
                            'https://www.nbcnews.com/select/shopping/mr-coffee-mug-warmer-gift-ncna1285511']
Length: 4, dtype: str

In [ ]:
# Look at articles with NAs in author that don't share URLs with the title NAs
nbc[~(nbc["title"].isna()) & (nbc["author"].isna())]["url"].values #visual inspection seems to indicate that these are all live updates

<StringArray>
[                                       'https://www.nbcnews.com/politics/supreme-court/live-blog/trump-immunity-supreme-court-ruling-live-updates-rcna159539',
                                      'https://www.nbcnews.com/politics/2024-election/live-blog/dnc-2024-live-updates-rcna165226/rcrd52541?canonicalCard=true',
                                        'https://www.nbcnews.com/politics/donald-trump/live-blog/live-updates-trump-indictment-classified-documents-rcna88494',
                                                    'https://www.nbcnews.com/politics/live-blog/jan-6-hearing-expect-top-trump-admin-aide-testifies-rcna35551',
                                          'https://www.nbcnews.com/news/world/live-blog/israel-hamas-war-live-updates-rcna141090/rcrd35599?canonicalCard=true',
                                          'https://www.nbcnews.com/news/world/live-blog/israel-hamas-war-live-updates-rcna134047/rcrd31099?canonicalCard=true',
                          

In [10]:
# Check if author NAs (excluding title NAs) are all live updates
print("Only 80 of the 92 articles with NA author are live updates, the next step is to examine the remaining 12.")
print("The below 80 articles can be handled together with one script that extracts all named contributors using meta tags in <head> into a comma separated list \nthat we can decide what to do with that later.")
nbc[~(nbc["title"].isna()) & (nbc["author"].isna()) & (nbc["url"].str.contains("live-updates"))] # Only 80 of 92 are live-updates, need to inspect remaining 12 
# -> these 80 can be handled together with one script
# Can extract all named contributors using meta tags in <head> into a comma separated list and then decide what to do with that later

Only 80 of the 92 articles with NA author are live updates, the next step is to examine the remaining 12.
The below 80 articles can be handled together with one script that extracts all named contributors using meta tags in <head> into a comma separated list 
that we can decide what to do with that later.


,url,topic,title,subtitle,author,datetime_posted,label
12,https://www.nbcnews.com/politics/supreme-court...,politics,Trump has some immunity in D.C. election inter...,Updates and the latest news coverage as Suprem...,NaN,NaN,NBC News
25,https://www.nbcnews.com/politics/2024-election...,politics,Coalition to March on the DNCâs protest ends...,While the situation that unfolded in Park 578 ...,NaN,NaN,NBC News
28,https://www.nbcnews.com/politics/donald-trump/...,politics,Trump indictment: Former president faces 37 co...,Latest news on former President Donald Trumpâ...,NaN,NaN,NBC News
68,https://www.nbcnews.com/news/world/live-blog/i...,news,IDF has been securing aid convoys for 4 nights...,Israel's military has been securing aid convoy...,NaN,NaN,NBC News
69,https://www.nbcnews.com/news/world/live-blog/i...,news,"Gaza hits 96-hour telecom blackout, a new record",The Gaza Strip has gone without substantial in...,NaN,NaN,NBC News
...,...,...,...,...,...,...,...
1711,https://www.nbcnews.com/news/world/live-blog/i...,news,U.S. urges Israel to end large-scale ground war,Despite some tough talk from President Joe Bid...,NaN,NaN,NBC News
1725,https://www.nbcnews.com/news/world/live-blog/i...,news,U.S. intel says it has âhigh confidenceâ I...,Follow live NBC News coverage of the Israel-Ha...,NaN,NaN,NBC News
1731,https://www.nbcnews.com/news/world/live-blog/u...,news,United Nations General Assembly: Biden gives f...,World leaders gather at the United Nations Gen...,NaN,NaN,NBC News
1754,https://www.nbcnews.com/news/world/live-blog/i...,news,"Hundreds of desperate Gazans rush aid truck, l...","In the dead of night, thousands of hungry Gaza...",NaN,NaN,NBC News


In [11]:
# Inspect remaining 12 urls that have NA author, non-NA title, and are not live updates
print("The remaining 12 URLs all seem to be different and will need to be handled individually.")
nbc[~(nbc["title"].isna()) & (nbc["author"].isna()) & ~(nbc["url"].str.contains("live-updates"))]["url"].values # These all seem to be different, will need to be handled individually

The remaining 12 URLs all seem to be different and will need to be handled individually.


<StringArray>
[                        'https://www.nbcnews.com/politics/live-blog/jan-6-hearing-expect-top-trump-admin-aide-testifies-rcna35551',
                      'https://www.nbcnews.com/politics/congress/blog/electoral-college-certification-updates-n1252864/ncrd1253052',
                                                    'https://www.nbcnews.com/politics/2024-primary-elections/florida-house-results',
                                      'https://www.nbcnews.com/specials/gaza-universities-destroyed-israel-military-war/index.html',
                                                  'https://www.nbcnews.com/politics/2024-primary-elections/minnesota-house-results',
                        'https://www.nbcnews.com/politics/donald-trump/live-blog/capitol-riot-january-6-committee-hearing-n1275078',
 'https://www.nbcnews.com/news/us-news/live-blog/columbia-protests-live-update-encampment-continue-college-negotiates-p-rcna149111',
                                                 'https

In [23]:
# Check datatime_posted NAs
nbc[~(nbc["title"].isna()) & (nbc["datetime_posted"].isna())]["url"].values # Seem to all be live blogs

<StringArray>
[                                       'https://www.nbcnews.com/politics/supreme-court/live-blog/trump-immunity-supreme-court-ruling-live-updates-rcna159539',
                                      'https://www.nbcnews.com/politics/2024-election/live-blog/dnc-2024-live-updates-rcna165226/rcrd52541?canonicalCard=true',
                                        'https://www.nbcnews.com/politics/donald-trump/live-blog/live-updates-trump-indictment-classified-documents-rcna88494',
                                                    'https://www.nbcnews.com/politics/live-blog/jan-6-hearing-expect-top-trump-admin-aide-testifies-rcna35551',
                                          'https://www.nbcnews.com/news/world/live-blog/israel-hamas-war-live-updates-rcna141090/rcrd35599?canonicalCard=true',
                                          'https://www.nbcnews.com/news/world/live-blog/israel-hamas-war-live-updates-rcna134047/rcrd31099?canonicalCard=true',
                          

In [25]:
# We know there are 80 live updates, so look at remaining 10 aritcles with datetime_posted NA
nbc[~(nbc["title"].isna()) & (nbc["datetime_posted"].isna()) & ~(nbc["url"].str.contains("live-updates"))] 

,url,topic,title,subtitle,author,datetime_posted,label
48,https://www.nbcnews.com/politics/live-blog/jan...,politics,Jan. 6 hearing highlights: What to expect as f...,The House committee investigating Jan. 6 held ...,NaN,NaN,NBC News
376,https://www.nbcnews.com/politics/congress/blog...,politics,"Defying Trump, Pence says he won't overturn th...",Vice President Mike Pence said in a letter rel...,NaN,NaN,NBC News
426,https://www.nbcnews.com/politics/2024-primary-...,politics,Florida House Primary Election 2024 Live Results,View the 2024 Florida primary election for Hou...,NaN,NaN,NBC News
672,https://www.nbcnews.com/specials/gaza-universi...,specials,Class destroyed: The rise and ruin of Gaza's r...,"Built over decades, Gazaâs universities embo...",NaN,NaN,NBC News
774,https://www.nbcnews.com/politics/2024-primary-...,politics,Minnesota House Primary Election 2024 Live Res...,View the 2024 Minnesota primary election for H...,NaN,NaN,NBC News
984,https://www.nbcnews.com/politics/donald-trump/...,politics,January 6 hearing: Capitol riot inquiry invoke...,Capitol police testified at the first hearing ...,NaN,NaN,NBC News
1472,https://www.nbcnews.com/news/us-news/live-blog...,news,Campus protests: Pro-Palestinian demonstration...,Gaza war protest encampments spread across U.S...,NaN,NaN,NBC News
1545,https://www.nbcnews.com/politics/2024-primary-...,politics,Minnesota Senate Primary Election 2024 Live Re...,View the 2024 Minnesota primary election for S...,NaN,NaN,NBC News
1626,https://www.nbcnews.com/politics/2024-primary-...,politics,Arizona Senate Primary Election 2024 Live Results,View the 2024 Arizona primary election for Sen...,NaN,NaN,NBC News
1724,https://www.nbcnews.com/politics/2024-primary-...,politics,Wisconsin House Primary Election 2024 Live Res...,View the 2024 Wisconsin primary election for H...,NaN,NaN,NBC News


In [7]:
# Verify: do the 10 non-live-update datetime NA articles overlap with the 12 non-live-update author NA articles?
dt_na_urls = set(nbc[~(nbc["title"].isna()) & (nbc["datetime_posted"].isna()) & ~(nbc["url"].str.contains("live-updates"))]["url"])
author_na_urls = set(nbc[~(nbc["title"].isna()) & (nbc["author"].isna()) & ~(nbc["url"].str.contains("live-updates"))]["url"])

exclusive_to_dt_na = dt_na_urls - author_na_urls
print(f"URLs with datetime NA but not author NA: {len(exclusive_to_dt_na)}")  # Expect 0

URLs with datetime NA but not author NA: 0


## Summary of NA Value Exploration for NBC Dataset

### Key Findings:

**Title NAs (4 articles):**
- All 4 articles with missing titles are pages that don't load
- Both title and subtitle are NA for the same 4 articles

**Author NAs (92 articles):**
- 80 of 92 articles with missing authors are live-update articles
- These 80 can be handled uniformly by extracting named contributors from meta tags in HTML `<head>` as comma-separated lists
- Remaining 12 articles with NA authors (non-live-update, non-NA title) are diverse and will need individual handling

**Datetime_posted NAs (90 articles):**
- All 80 live-update articles have missing datetime_posted values
- Remaining 10 articles with missing datetime_posted (non-live-update, non-NA title) overlap completely with the 12 articles that have NA author values
- These articles will need individual investigation

### Data Quality Recommendations:
1. Remove or flag the 4 articles with missing titles (pages that don't load)
2. Handle the 80 live-update articles uniformly using meta tag extraction
3. Individually process the 12 remaining problematic articles that have both missing author and datetime_posted values
